<a href="https://colab.research.google.com/github/aditya03bairagi/Data-Science-Projects/blob/projects/Customer_Churn_Analysis_Data_Science_Project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Customer Churn Analysis & Insights Dashboard

## Project Objectives

This project focuses on exploratory analysis and visual storytelling to drive business decisions related to customer churn.

*   **Perform exploratory data analysis (EDA)** to uncover churn patterns by segment.
*   **Build interactive dashboards** to visualize key metrics like retention rate and lifetime value (through static plots using Matplotlib).
*   **Identify top features correlated with churn** and present actionable business recommendations.

## Part 1: Data Loading and Initial Inspection

Since no specific dataset was provided, we will generate a synthetic dataset that simulates customer churn data. This dataset will include demographic information, service usage details, and billing information.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a random seed for reproducibility
np.random.seed(42)

# Number of customers
n_customers = 5000

# Generate synthetic data
data = {
    'customerID': [f'C{i:05d}' for i in range(n_customers)],
    'gender': np.random.choice(['Male', 'Female'], n_customers),
    'SeniorCitizen': np.random.choice([0, 1], n_customers, p=[0.8, 0.2]),
    'Partner': np.random.choice(['Yes', 'No'], n_customers),
    'Dependents': np.random.choice(['Yes', 'No'], n_customers),
    'tenure': np.random.randint(1, 73, n_customers), # Months
    'PhoneService': np.random.choice(['Yes', 'No'], n_customers),
    'MultipleLines': np.random.choice(['Yes', 'No', 'No phone service'], n_customers, p=[0.45, 0.45, 0.1]),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_customers, p=[0.35, 0.45, 0.2]),
    'OnlineSecurity': np.random.choice(['Yes', 'No', 'No internet service'], n_customers, p=[0.25, 0.55, 0.2]),
    'OnlineBackup': np.random.choice(['Yes', 'No', 'No internet service'], n_customers, p=[0.3, 0.5, 0.2]),
    'DeviceProtection': np.random.choice(['Yes', 'No', 'No internet service'], n_customers, p=[0.25, 0.55, 0.2]),
    'TechSupport': np.random.choice(['Yes', 'No', 'No internet service'], n_customers, p=[0.25, 0.55, 0.2]),
    'StreamingTV': np.random.choice(['Yes', 'No', 'No internet service'], n_customers, p=[0.35, 0.45, 0.2]),
    'StreamingMovies': np.random.choice(['Yes', 'No', 'No internet service'], n_customers, p=[0.35, 0.45, 0.2]),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_customers, p=[0.55, 0.25, 0.2]),
    'PaperlessBilling': np.random.choice(['Yes', 'No'], n_customers),
    'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], n_customers),
    'MonthlyCharges': np.random.uniform(18, 120, n_customers).round(2)
}

df = pd.DataFrame(data)

# TotalCharges: MonthlyCharges * tenure, with some random noise and some missing values
df['TotalCharges'] = df.apply(lambda row: row['MonthlyCharges'] * row['tenure'] * np.random.uniform(0.9, 1.1) if row['tenure'] > 0 else 0, axis=1).round(2)
# Introduce some missing values for TotalCharges to simulate real data
missing_indices = np.random.choice(df.index, int(0.01 * n_customers), replace=False)
df.loc[missing_indices, 'TotalCharges'] = np.nan

# Churn: Generate based on factors like contract, tenure, internet service, and monthly charges
# Customers with month-to-month contracts, higher monthly charges, shorter tenure, and fiber optic are more likely to churn
churn_prob = (
    0.3 * (df['Contract'] == 'Month-to-month').astype(int) +
    0.2 * (df['InternetService'] == 'Fiber optic').astype(int) +
    0.1 * (df['tenure'] < 12).astype(int) +
    0.1 * (df['MonthlyCharges'] > 70).astype(int) +
    np.random.uniform(-0.2, 0.2, n_customers) # Random noise
)
churn_prob = np.clip(churn_prob, 0.05, 0.95) # Clip probabilities to a reasonable range
df['Churn'] = (np.random.rand(n_customers) < churn_prob).astype(int)

print("Synthetic dataset created successfully!")

Synthetic dataset created successfully!


In [ ]:
# Display the first 5 rows of the dataset
display(df.head())

# Display basic information about the dataset
df.info()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,C00000,Male,1,No,Yes,62,Yes,Yes,No,No,...,No,No internet service,No,No internet service,Month-to-month,Yes,Electronic check,35.41,2374.52,0
1,C00001,Female,0,No,No,69,No,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),100.47,7328.31,1
2,C00002,Male,0,No,No,63,No,Yes,Fiber optic,No,...,Yes,Yes,No,No,Month-to-month,Yes,Electronic check,110.99,7025.98,1
3,C00003,Male,1,Yes,No,2,No,Yes,DSL,No internet service,...,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),109.83,201.82,1
4,C00004,Male,0,Yes,Yes,54,No,Yes,Fiber optic,No,...,Yes,Yes,No,Yes,Month-to-month,No,Mailed check,53.35,3137.56,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5000 non-null   object 
 1   gender            5000 non-null   object 
 2   SeniorCitizen     5000 non-null   int64  
 3   Partner           5000 non-null   object 
 4   Dependents        5000 non-null   object 
 5   tenure            5000 non-null   int64  
 6   PhoneService      5000 non-null   object 
 7   MultipleLines     5000 non-null   object 
 8   InternetService   5000 non-null   object 
 9   OnlineSecurity    5000 non-null   object 
 10  OnlineBackup      5000 non-null   object 
 11  DeviceProtection  5000 non-null   object 
 12  TechSupport       5000 non-null   object 
 13  StreamingTV       5000 non-null   object 
 14  StreamingMovies   5000 non-null   object 
 15  Contract          5000 non-null   object 
 16  PaperlessBilling  5000 non-null   object 


In [ ]:
# Display descriptive statistics for numerical columns
display(df.describe())

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
count,5000.000000,5000.000000,5000.000000,4950.000000,5000.000000
count,5000.000000,5000.000000,5000.000000,4950.000000,5000.000000
mean,0.199000,36.687200,68.817892,2522.571527,0.341000
std,0.399288,20.817023,29.690465,1905.339968,0.474093
min,0.000000,1.000000,18.010000,19.380000,0.000000
25%,0.000000,18.000000,43.197500,978.650000,0.000000
min,0.000000,1.000000,18.010000,19.380000,0.000000
50%,0.000000,37.000000,68.660000,2053.975000,0.000000
75%,0.000000,55.000000,95.025000,3719.790000,1.000000
max,1.000000,72.000000,119.990000,9031.070000,1.000000


## Part 2: Data Preprocessing and Cleaning

In [ ]:
# Handle missing values in 'TotalCharges'
# First, convert 'TotalCharges' to numeric, coercing errors will turn non-numeric values into NaN
# Then, fill NaN values with the median of the column
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Convert 'SeniorCitizen' to object type for consistency with other categorical features
df['SeniorCitizen'] = df['SeniorCitizen'].astype(object)

# Convert 'Churn' to categorical type for analysis
df['Churn'] = df['Churn'].astype(object)

print("Missing values in 'TotalCharges' handled and data types adjusted.")

Missing values in 'TotalCharges' handled and data types adjusted.


In [ ]:
# Display basic information about the dataset after preprocessing
df.info()

# Display descriptive statistics again to see changes
display(df.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5000 non-null   object 
 1   gender            5000 non-null   object 
 2   SeniorCitizen     5000 non-null   object 
 3   Partner           5000 non-null   object 
 4   Dependents        5000 non-null   object 
 5   tenure            5000 non-null   int64  
 6   PhoneService      5000 non-null   object 
 7   MultipleLines     5000 non-null   object 
 8   InternetService   5000 non-null   object 
 9   OnlineSecurity    5000 non-null   object 
 10  OnlineBackup      5000 non-null   object 
 11  DeviceProtection  5000 non-null   object 
 12  TechSupport       5000 non-null   object 
 13  StreamingTV       5000 non-null   object 
 14  StreamingMovies   5000 non-null   object 
 15  Contract          5000 non-null   object 
 16  PaperlessBilling  5000 non-null   object 
